# Demo 1: Classify and prepare temporal structure

**Learning question:** How do grain, entity, ordering, spacing, and timezone meaning determine a safe temporal table?

The input and output grain are both **one recorded station temperature observation per row**. The output preserves all supplied rows but gives each timestamp an unambiguous UTC meaning and a deliberate within-entity order. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define the temporal structure before parsing

A **timestamp** represents an instant; a **period** represents a span with a start and end. An **entity** is the real-world unit with one ordered history, and an **entity key** identifies it. A **single series** contains one entity history; a **panel** contains multiple entity histories.

**Row grain** states what one source row represents. A **row key** identifies one row, while **sort keys** state the order required for computation. Here one row is one recorded temperature observation, `station` is the entity key, `station,observed_at` is the row key, and the sort keys are station then timestamp.

A series is **regular** when adjacent observations follow one expected spacing and **irregular** when the gaps vary. A **frequency** names an expected grid or calendar offset. The current pandas hourly alias is lowercase `h`.

**Parsing** converts text to datetime values. A **naive timestamp** has no timezone offset. A **timezone-aware timestamp** identifies an offset and therefore an unambiguous instant. **Localization** attaches the documented source zone without changing its displayed clock reading; **conversion** expresses an already-aware instant in another zone. A **DatetimeIndex** is an index whose labels are datetime values.

Prediction before code: this fixture is timestamp-based, irregular, and a two-entity panel. Repeated timestamps across stations are valid because the row key includes `station`.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "57dcdb82372805cf1dda83a7c227b463fe997cf1437275d64d01b9719ff26b54"
FIXTURE_BYTES = (
    b"station,observed_at,temperature_c\n"
    b"south,2026-01-15 13:00,23.0\n"
    b"north,2026-01-15 08:00,10.0\n"
    b"south,2026-01-15 08:00,20.0\n"
    b"north,2026-01-15 14:00,14.0\n"
    b"south,2026-01-15 10:00,21.0\n"
    b"north,2026-01-15 11:00,\n"
    b"south,2026-01-15 14:00,24.0\n"
    b"north,2026-01-15 09:00,11.0\n"
    b"south,2026-01-15 11:00,22.0\n"
    b"north,2026-01-15 12:00,13.0\n"
)


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "09" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "station_observations.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "station_observations.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

raw = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
    },
)
raw["source_row"] = np.int64(1)

assert raw.shape == (10, 4)
assert raw["station"].dtype == pd.StringDtype()
assert raw["observed_at"].dtype == pd.StringDtype()
assert raw["temperature_c"].dtype == np.dtype("float64")
assert raw["source_row"].dtype == np.dtype("int64")
assert raw["temperature_c"].isna().sum() == 1

naive_times = pd.to_datetime(
    raw["observed_at"],
    format="%Y-%m-%d %H:%M",
)
assert naive_times.dt.tz is None
aware_times = naive_times.dt.tz_localize("America/Los_Angeles")
raw["observed_at"] = aware_times.dt.tz_convert("UTC")

prepared = raw.sort_values(
    ["station", "observed_at"],
    kind="stable",
).reset_index(drop=True)

assert prepared["station"].dtype == pd.StringDtype()
assert str(prepared["observed_at"].dtype) == "datetime64[us, UTC]"
assert prepared["temperature_c"].dtype == np.dtype("float64")
assert prepared["source_row"].dtype == np.dtype("int64")
assert prepared.shape == (10, 4)
assert not prepared.duplicated(["station", "observed_at"]).any()
assert all(
    group["observed_at"].is_monotonic_increasing
    for _, group in prepared.groupby(
        "station", observed=True, sort=True, dropna=True
    )
)


OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OWNED_OUTPUT_NAMES = ['prepared_panel.csv']
for output_name in OWNED_OUTPUT_NAMES:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()


def write_verified_csv(frame, path, *, expected_size, expected_sha256):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    first_bytes = path.read_bytes()
    assert len(first_bytes) == expected_size
    assert sha256(first_bytes).hexdigest() == expected_sha256
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


def restore_utc_timestamp(frame, column="observed_at"):
    frame[column] = pd.to_datetime(
        frame[column],
        format="%Y-%m-%d %H:%M:%S%z",
        utc=True,
    )
    return frame


print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)


In [ ]:
example_timestamp = pd.Timestamp("2026-01-15 08:00")
example_period = pd.Period("2026-01-15", freq="D")
assert example_timestamp == pd.Timestamp("2026-01-15 08:00:00")
assert example_period.start_time == pd.Timestamp("2026-01-15 00:00:00")
assert example_period.end_time.date().isoformat() == "2026-01-15"

regular_reference = pd.date_range(
    "2026-01-15 16:00",
    periods=4,
    freq="h",
    tz="UTC",
)
regular_gaps = regular_reference.to_series().diff().dropna()
assert regular_gaps.eq(pd.Timedelta(hours=1)).all()

gaps = prepared.groupby(
    "station",
    observed=True,
    sort=True,
    dropna=True,
)["observed_at"].diff()
assert sorted(gaps.dropna().unique()) == [
    pd.Timedelta(hours=1),
    pd.Timedelta(hours=2),
]
expected_gap_hours = {
    "north": [1.0, 2.0, 1.0, 2.0],
    "south": [2.0, 1.0, 2.0, 1.0],
}
for station, group in prepared.groupby(
    "station", observed=True, sort=True, dropna=True
):
    station_gap_hours = (
        group["observed_at"].diff().dropna().dt.total_seconds() / 3600
    )
    assert station_gap_hours.tolist() == expected_gap_hours[station], station

indexed_panel = prepared.set_index("observed_at")
assert isinstance(indexed_panel.index, pd.DatetimeIndex)
assert str(indexed_panel.index.tz) == "UTC"
assert "station" in indexed_panel.columns
assert indexed_panel["station"].nunique(dropna=True) == 2

north_series = indexed_panel.loc[indexed_panel["station"].eq("north")]
assert len(north_series) == 5
assert north_series.index.is_monotonic_increasing
assert prepared["observed_at"].duplicated(keep=False).sum() == 6
assert prepared["station"].tolist() == ["north"] * 5 + ["south"] * 5

print("Regular reference gaps:", regular_gaps.tolist())
print("Panel gaps by station:", gaps.tolist())
print(prepared)


In [ ]:
PREPARED_OUTPUT_PATH = OUTPUT_DIRECTORY / "prepared_panel.csv"
prepared_bytes = write_verified_csv(
    prepared,
    PREPARED_OUTPUT_PATH,
    expected_size=431,
    expected_sha256="a9e2b75c2f4e9f9a3778b53cd87e68d4a559511c368cad80b7153b1109a987ba",
)

prepared_readback = pd.read_csv(
    PREPARED_OUTPUT_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
        "source_row": "int64",
    },
)
restore_utc_timestamp(prepared_readback)
assert prepared_readback["station"].dtype == pd.StringDtype()
assert str(prepared_readback["observed_at"].dtype) == "datetime64[us, UTC]"
assert prepared_readback["temperature_c"].dtype == np.dtype("float64")
assert prepared_readback["source_row"].dtype == np.dtype("int64")
pd.testing.assert_frame_equal(prepared_readback, prepared)

demo1_verified = True
print("Wrote:", PREPARED_OUTPUT_PATH)
print("Output SHA-256:", sha256(prepared_bytes).hexdigest())


## Interpret the prepared panel

Both stations contain one- and two-hour gaps, so neither history is regular. The North-only view is one single series; the complete table remains a panel. Sorting by station and time preserves each entity boundary. A timestamp duplicated across stations is not a duplicated row because `station,observed_at` is the row key.


In [ ]:
assert demo1_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert PREPARED_OUTPUT_PATH.is_file()
assert len(PREPARED_OUTPUT_PATH.read_bytes()) == 431
assert sha256(PREPARED_OUTPUT_PATH.read_bytes()).hexdigest() == "a9e2b75c2f4e9f9a3778b53cd87e68d4a559511c368cad80b7153b1109a987ba"
assert prepared.shape == (10, 4)
assert prepared.groupby(
    "station", observed=True, sort=True, dropna=True
).ngroups == 2
print("Lecture 09 Demo 1 fresh-execution verification passed.")
